# __Lecture 08: Practical analyses in Python__

----
### **_Be sure to pull the latest notebook!!_**

#### __Announcement:__ Homework 3 is due on October 27

#### __Last week, we talked about:__
- Python! (e.g. data types, lists, dictionaries, for loops, if statements, functions)

#### __Today we will:__
- warm up by writing a function that uses a dictionary
- talk about why __searching for patterns__ comes up constantly in computational biology
- introduce regular expressions and the `re` module
- write __tests__ for our pattern-matching code, and use them to catch a bug

###  
----

-----
# __Warm-up: defining a function__

Everything we do today builds on the Python from lectures 6 and 7: dictionaries, `for` loops, `if`
statements, and functions with default arguments. Let's warm up with all four at once.

#### Write a function that uses a dictionary to count the number of times each base occurs.

Be sure to account for sequences that contain uppercase and lowercase bases. I've provided an empty
function here (with the docstring already written). Use the provided test cases to check your work.

__A note on docstrings:__ in this class we write [Google-style docstrings](https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings),
like the one below: a one-line summary, then `Args:`, then `Returns:`. Getting in the habit of
writing the docstring *before* the code is a good way to force yourself to decide what the function
is actually supposed to do.

In [ ]:
def count_bases(seq, case="upper"):
    """Count the number of times each base occurs in a sequence.

    Args:
        seq: DNA sequence, as a string.
        case: Case in which to count the bases, either "upper" or "lower".
            Defaults to "upper".

    Returns:
        A dict keyed by each nucleotide, where the value is the number of times
        that nucleotide is observed in the sequence.

    Note:
        <object>.upper() will convert the <object> string to uppercase.
    """

    assert case in ["lower", "upper"], "case needs to be either lower or upper"

    base_counts = {}

    if case == "upper":
        new_seq = seq.upper()
    elif case == "lower":
        new_seq = seq.lower()

    for base in new_seq:
        if base in base_counts:
            base_counts[base] += 1
        else:
            base_counts[base] = 1

    return base_counts

In [ ]:
count_bases("AATXXGGCT")

{'A': 2, 'T': 2, 'X': 2, 'G': 2, 'C': 1}

In [ ]:
count_bases("aatTGGcT", case="lower")

{'a': 2, 't': 3, 'g': 2, 'c': 1}

In [ ]:
count_bases("ATGCG", case="unicorn")

AssertionError: case needs to be either lower or upper

----
# __Searching for patterns__

As we start working with sequencing data, we will want to be able to search sequences for patterns
of interest.

More broadly: a surprising amount of computational biology is __text parsing__. Sequencing machines,
alignment tools, genome annotations, and your own lab's file naming conventions all produce text
that is not quite a tidy table, but is not random either. It has a __shape__, and that shape usually
tells you something about the experiment.

Here are a few examples you are likely to run into:

In [ ]:
# 1. An influenza strain name downloaded from a sequence database.
#    Q: what year was it collected? What subtype is it?
strain = "A/New York/3/1994 (H3N2)"

# 2. A read name from an Illumina sequencer (we'll see these again on Thursday).
#    Q: which lane and tile did this read come from?
read_name = "@HISEQ:621:HMJGNBCX2:1:1101:1797:2150 1:N:0:ATGTCA"

# 3. FASTQ file names from a shared directory, named by three different people.
#    Q: which gene and which replicate is each file?
filenames = [
    "Sample_01_HA_rep1_R1.fastq.gz",
    "sample-02-NA-rep2_R1.fastq.gz",
    "S03.HA.rep1.R1.fastq.gz",
]

print(strain)
print(read_name)
print(filenames)

Last week, we learned that you can search for a substring within a string like this:

In [ ]:
seq = "ATTCCGAAA"
subseq = "GAA"

# we can use the in keyword
subseq in seq

True

The `in` keyword answers a __yes/no__ question about an __exact__ piece of text, which means you
have to already know what you are looking for. That is often not the situation we are in:

In [ ]:
# `in` can confirm a subtype we already know about...
print("H3N2" in strain)

# ...but it can't tell us the subtype of a strain we haven't seen before
mystery_strain = "A/swine/Iowa/15/1930 (H1N1)"
print("H3N2" in mystery_strain)  # False -- but there IS a subtype here, we just don't know it yet

# and .split() only helps when the text is perfectly consistent
print("Sample_01_HA_rep1_R1.fastq.gz".split("_"))
print("sample-02-NA-rep2_R1.fastq.gz".split("_"))

### What does it mean to search for a pattern "algorithmically"?

Before we reach for a new module, it is worth seeing that we could do this ourselves with the Python
we already have. Say we expect sequences that look like `CTAGC`-`NNNNNN`-`GATCA`, where the six `N`s
are a barcode. To find the barcode, we have to do three things:

1. __scan__: step along the sequence, one position at a time
2. __test__: at this position, does the text match what we expect?
3. __report__: if it does, pull out the piece we care about

In [ ]:
def find_barcode_by_hand(seq, upstream="CTAGC", downstream="GATCA", bclen=6):
    """Find a barcode flanked by fixed sequences, by checking every position.

    Args:
        seq: DNA sequence, as a string.
        upstream: Sequence expected immediately upstream of the barcode.
        downstream: Sequence expected immediately downstream of the barcode.
        bclen: Length of the barcode.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected design.
    """
    seq = seq.upper()
    construct_len = len(upstream) + bclen + len(downstream)

    # SCAN: try every position where the whole construct could still fit
    for i in range(len(seq) - construct_len + 1):
        # TEST: is the upstream sequence here?
        if seq[i:i + len(upstream)] != upstream:
            continue

        bc_start = i + len(upstream)
        bc_end = bc_start + bclen

        # TEST: is the downstream sequence exactly bclen bases later?
        if seq[bc_end:bc_end + len(downstream)] != downstream:
            continue

        # REPORT
        return seq[bc_start:bc_end]

    return None


print(find_barcode_by_hand("CTAGCatcgatGATCA"))
print(find_barcode_by_hand("CCAGCatagcaGATCA"))

That works! So why do we need anything else?

Because the __scan__ step is always the same ("walk along the string and try each position"), but the
__test__ step changes every time the experiment does. Suppose the upstream sequence could be `CTAGC`
*or* `CTGGC`, the barcode could be 6 *to* 8 bases long, and we only want to accept `A`, `C`, `G`, `T`
and `N` in the barcode. All of that is doable with more loops and more `if` statements, but it gets
long and hard to change.

A __regular expression__ is a compact way of writing down just the __test__: you describe the shape
you are looking for, and the `re` module does the scanning and extracting for you.

---
# __New module:__  Regular expressions

Python has an extensive collection of built-in __modules__ containing special purpose functions and
objects - we have already used `math`, and on Thursday we will use `biopython`. Today's is `re`.
([The full list of built-in modules is here](https://docs.python.org/3/library/).)

#### A __regular expression__ is a sequence of characters that forms a __search pattern__.
They can be used to check if a string contains the specified search pattern.
This is particularly useful when parsing strings.

The `re` module offers a set of functions that allows us to search a string for a match using
"regular expressions". The main function we will be using is called `search`, which finds a match at
any position in the string. We can search for a pattern in a string like this:

In [ ]:
import re

# This is an example string
seq = "ATTCCGAAA"

# compile the search pattern
search_pattern = re.compile("GAA")

# use the search pattern to search the sequence
match = search_pattern.search(seq)

print(match)

# extract your match
print(match.group())

<re.Match object; span=(5, 8), match='GAA'>
GAA


However, the `re` module allows us to define much more general search patterns. The tables below
cover almost everything you will need.

#### Matching specific characters

| pattern | matches |
| --- | --- |
| `A`, `7`, `-` | that exact character |
| `\.` `\$` `\(` `\)` `\*` `\+` | a *literal* `.` `$` `(` `)` `*` `+` - characters that have a special meaning below must be escaped with `\` |

#### Matching a class of characters

| pattern | matches |
| --- | --- |
| `.` | any single character |
| `\d` | any digit, `0`-`9` |
| `\w` | any "word" character: a letter, a digit, or an underscore |
| `\s` | any whitespace character |
| `[ACGTN]` | any one of `A`, `C`, `G`, `T`, `N` (i.e. any one nucleotide) |
| `[a-n]` | any one lower-case letter from `a` to `n` |
| `[^ACGT]` | any character that is *not* `A`, `C`, `G`, or `T` |

#### How many times

| pattern | matches |
| --- | --- |
| `a+` | one or more `a` |
| `a*` | zero or more `a` |
| `a?` | zero or one `a` |
| `a{6}` | exactly six `a` |
| `a{6,8}` | between six and eight `a` |

#### Where in the string

| pattern | matches |
| --- | --- |
| `^A` | an `A` at the *beginning* of the string |
| `A$` | an `A` at the *end* of the string |

#### Grouping and naming

| pattern | meaning |
| --- | --- |
| `(...)` | group this part of the pattern together, and capture what it matched |
| `(?P<name>...)` | capture it *and* give it a name, so you can ask for it by name later |
| `a\|b` | match `a` *or* `b` |

#### Putting the pieces together

| pattern | reads as |
| --- | --- |
| `\d{4}` | four digits in a row |
| `^ATG` | a sequence that starts with `ATG` |
| `[ACGT]{6,8}` | six to eight nucleotides |
| `(?P<year>\d{4})` | four digits, captured under the name `year` |

__A note on quotes:__ you will see search patterns written as _raw strings_, like `r"\d{4}"`. The `r`
tells Python to leave the backslashes alone instead of interpreting them (`"\n"`, for example, is a
newline rather than a backslash and an `n`). Regular expressions use backslashes constantly, so
__always write your patterns as `r"..."`__ - it costs nothing and avoids a confusing class of bug.

There are many more special codes in the Python regular expression module (see
[here](https://docs.python.org/3/library/re.html)), and you can use them to do almost any type of
string matching.

You can test your regular expressions using [regex101.com](https://regex101.com), which shows you
piece by piece what a pattern is doing. Coding assistants like ChatGPT and Claude are also quite good
at writing regular expressions. That is genuinely useful - but it makes the other half of the job
more important, not less: __you__ still have to say precisely what the pattern should match, and
__you__ still have to check that it does. We will practice exactly that later in this notebook.

#### Example:
Here is a common example dealing with influenza.
You download some strains from the database, and they have names that look like this:

In [ ]:
strain = "A/New York/3/1994 (H3N2)"

Say you want to get some information out of these, like the year. Let's build a regular expression that gets the year out of `strain`:

In [27]:
import re

# compile a re for the year
yearmatch = re.compile(r"\d{4}")

# search for the search pattern in the string
match = yearmatch.search(strain)

# return our search pattern match
match

<re.Match object; span=(13, 17), match='1994'>

#### Adding __groups__ and __names__ to search patterns

Sometimes you may want to group parts of a search pattern or create named groups for easier reference. Here's how:

* **Grouping patterns**: `()` is used to group part of your search pattern. Everything inside the parentheses is treated as a single unit, and you can capture and reference this group.
* **Named groups**: `(?P<name>...)` defines a named group with the specified name, allowing you to reference this part of the match later (e.g., `(?P<name>\d{4})` for a four-digit year).

In [ ]:
# compile a named re for the year
## here, we are searching for 4 digits that are preceded by the / character
named_yearmatch = re.compile(r"\/(?P<year>\d{4})")

# search for the search pattern in the string
named_search = named_yearmatch.search(strain)

print(named_search)

<re.Match object; span=(12, 17), match='/1994'>


In [ ]:
# isolate named pattern
year = named_search.group("year")

print(year)

1994


----
### __Practice:__ 

#### Let's build a regular expression that gets the subtype (e.g. `H3N2`) out the `strain` object

_Hints:_
- _Be sure to name the search pattern `subtype`_
- _Include the parentheses in your search pattern, but not in the final named search pattern_

In [36]:
pattern = re.compile(r"\((?P<subtype>[A-Z]\d+[A-Z]\d+)\)")

pattern_match = pattern.search(strain)

print(pattern_match)

print(pattern_match.group("subtype"))

<re.Match object; span=(18, 24), match='(H3N2)'>
H3N2


----
### __Practice:__

#### Using regular expressions to parse barcodes

Now we will use regular expressions to parse barcodes from nucleotide sequences.
For instance, you might have to do this in a single-cell RNA-seq experiment where there is a barcode at the end of each read telling you the cell that the read came from.

Imagine that our valid molecules should have sequences like this:

`CTAGCNNNNNNGATCA`

See how there is a 6-nucleotide barcode in the center of the sequence.
We have a list of sequences, and want to parse through them to figure out which ones meet the expected pattern--and get the barcode from those that do:

In [38]:
seqs = [
    "CTAGCatcgatGATCA",  # has barcode ATCGAT
    "CCAGCatagcaGATCA",  # does not have expected 5' sequence
    "CTAGCtacagGATCA",  # barcode too short
    "CTAGCgaccatGATCA",  # has barcode GACCAT
    "CTAGCatcgatGATCA",  # has barcode ATCGAT
    "CTAGCatcgatGGTCA",  # does not have expected 3' sequence
]

#### __Part 1__
First, take a few minutes in groups to write a search pattern that will match the valid barcodes from these sequences.
When doing this, note that:

- You will want to __group__ and __name__ the barcode portion of the search pattern
- You will want to __include the fixed upstream (CTAGC) and downstream (GATCA) portions__ of the sequence in your search pattern, but not as part of the barcode subgroup

In [39]:
pat = re.compile("CTAGC(?P<barcode>[A-Z]{6})GATCA")

for seq in seqs:
    upperseq = seq.upper()
    m = pat.search(upperseq)
    if m is None:
        print("no match")
    else:
        print(m.group("barcode"))

ATCGAT
no match
no match
GACCAT
ATCGAT
no match


----
# __Checking that a search pattern is right__

Here is an uncomfortable fact about regular expressions: __a wrong pattern looks exactly like a right
one__. It does not crash, and it does not warn you. It quietly returns answers, and those answers end
up in your counts and your figures.

Patterns also tend to come from somewhere else - a script you inherited from a previous lab member,
an answer you found online, something a coding assistant wrote for you in a few seconds. That is
fine. It just means the valuable skill is being able to say what the pattern __should__ do, and then
check that it does.

#### There are two ways a pattern can be wrong

| failure | what it means | what it looks like in your data |
| --- | --- | --- |
| __too permissive__ | the pattern matches things it should not | junk silently counted as real data |
| __too strict__ | the pattern misses things it should match | real data thrown away; suspiciously low yield |

Both are invisible unless you go looking. So let's go looking, using the barcode pattern we just
wrote.

#### Step 1: write down what "correct" means, as examples

A __test case__ is one input paired with the answer we expect. Notice that good test sets include
__negatives__ - inputs that should return nothing - because a pattern that is too permissive can only
be caught by a negative case.

In [ ]:
# each entry is (input sequence, expected barcode)
test_cases = [
    ("CTAGCatcgatGATCA", "ATCGAT"),  # a good sequence, lower-case barcode
    ("CTAGCGACCATGATCA", "GACCAT"),  # a good sequence, upper case
    ("CCAGCatagcaGATCA", None),      # wrong upstream sequence
    ("CTAGCatcgatGGTCA", None),      # wrong downstream sequence
    ("CTAGCtacagGATCA", None),       # barcode is only 5 bases
]

#### Step 2: write a function that runs the test cases for you

The `assert` statement is the simplest way to check something, but it stops at the first failure,
which is unfriendly when you are debugging. Here is a small helper that runs __every__ case and
reports all of the failures at once.

In [ ]:
def check(parse_function, cases):
    """Run a parsing function against test cases and report any failures.

    Args:
        parse_function: Function that takes one sequence and returns a barcode
            string, or None if the sequence does not match.
        cases: List of (sequence, expected_result) tuples.

    Returns:
        A list of the cases that failed, as (sequence, expected, actual) tuples.
    """
    failures = []
    for sequence, expected in cases:
        actual = parse_function(sequence)
        if actual != expected:
            failures.append((sequence, expected, actual))

    if failures:
        print(f"FAILED {len(failures)} of {len(cases)} cases:")
        for sequence, expected, actual in failures:
            print(f"  {sequence}: expected {expected}, got {actual}")
    else:
        print(f"passed all {len(cases)} cases")

    return failures

#### Step 3: test the pattern we wrote above

Here is the same pattern from the practice above, wrapped in a function so that we can hand it to
`check`.

In [ ]:
pattern_v1 = re.compile(r"CTAGC(?P<barcode>[A-Z]{6})GATCA")


def parse_barcode_v1(seq):
    """Find the barcode in a sequence.

    Args:
        seq: DNA sequence, as a string.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected pattern.
    """
    match = pattern_v1.search(seq.upper())
    if match is None:
        return None
    return match.group("barcode")


check(parse_barcode_v1, test_cases)

----
### __Practice:__ break it

It passes everything - and this is exactly where people stop.

__Take a few minutes in groups.__ Find sequences where `parse_barcode_v1` gives the __wrong__ answer,
and add them to the test set as new `(sequence, expected)` cases. You are trying to make it fail.

_Hint 1: look hard at `[A-Z]`. Which characters does that actually allow? Are all of them nucleotides?_

_Hint 2: remember what `.search()` does. What happens to a sequence with extra junk stuck on the front?_

In [ ]:
# add your tricky cases here, then run check() again
tricky_cases = [
    # [A-Z] allows all 26 letters, so non-nucleotide characters sail right through
    ("CTAGCATXGATGATCA", None),

    # .search() finds the pattern anywhere, even when the sequence clearly isn't our construct
    ("GGGGCTAGCATCGATGATCATTTT", None),
]

check(parse_barcode_v1, test_cases + tricky_cases)

#### Step 4: fix the pattern, and keep the tests

Two small changes fix both problems:

- `[A-Z]` becomes `[ACGTN]`, so that the barcode can only contain things that are actually bases
- `^` is added at the front, so that the pattern has to match from the beginning of the sequence

This is the payoff of having written the tests: we can check the fix immediately, and the test cases
stay in the notebook. The next time someone edits this pattern, the tricky cases are still here,
still checking.

In [ ]:
pattern_v2 = re.compile(r"^CTAGC(?P<barcode>[ACGTN]{6})GATCA")


def parse_barcode_v2(seq):
    """Find the barcode in a sequence.

    Args:
        seq: DNA sequence, as a string.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected pattern.
    """
    match = pattern_v2.search(seq.upper())
    if match is None:
        return None
    return match.group("barcode")


check(parse_barcode_v2, test_cases + tricky_cases)

#### __Part 2__

Now, modify your code to create a function that parses barcoded sequences, extracts those with valid barcodes, and keeps track of the counts associated with each barcode (using a dictionary). 
This requires setting up your function so that it does the following:

 1. Converts each sequence to uppercase (using `s.upper()`)
 2. Searches the string for a specified search pattern
 3. Determines if it matches the expected pattern (with the correct length barcode and constant upstream/downstream sequences)
 3. If it matches, extracts the barcode and add it to a dictionary to keep track of counts

A few notes to keep in mind:
- You may want to search from the start of the string (use the `^` symbol in your search pattern).
- The barcode length, upstream sequence, and downstream sequence should be passed as function arguments with default values, rather than being fixed. You’ll need to generalize your search pattern from above to accommodate these variable inputs (e.g., by adding strings and variables together).

I’ve provided the function documentation below—try to implement it.
__Take a few minutes to work through this in groups.__

In [ ]:
def count_barcodes(seqs, bclen=6, upstream="CTAGC", downstream="GATCA"):
    """Parse and count barcodes.

    Args:
        seqs: List of DNA sequences.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode.
        downstream: Sequence expected immediately downstream of the barcode.

    Returns:
        A dict keyed by each valid barcode, where the value is the number of
        times that barcode is observed.

    Note:
        The function is **not** case-sensitive, and all barcodes are reported
        in upper case.
    """

    counts = {}

    # note the ^ and the [ACGTN], for the reasons we worked out above
    pat = re.compile("^" + upstream + "(?P<barcode>[ACGTN]{" + str(bclen) + "})" + downstream)

    for seq in seqs:
        upperseq = seq.upper()

        match = pat.search(upperseq)

        if match is None:
            continue

        barcode = match.group("barcode")

        if barcode in counts:
            counts[barcode] = counts[barcode] + 1
        else:
            counts[barcode] = 1

    return counts

Run the function once you've implemented it. Does it give the right result?

In [43]:
count_barcodes(seqs)

{'ATCGAT': 2, 'GACCAT': 1}

#### Test it before you trust it

We know what the answer should be here: `ATCGAT` appears twice, `GACCAT` appears once, and three
sequences should be rejected. Checking a known-answer case like this is the cheapest form of testing
there is, and it is worth doing every single time.

In [ ]:
counts = count_barcodes(seqs)
print(counts)

assert counts["ATCGAT"] == 2
assert counts["GACCAT"] == 1
assert len(counts) == 2, "we should only have found two distinct barcodes"

# and a check that the arguments actually do something: parsing these same sequences
# with a DIFFERENT expected design should give us nothing
assert len(count_barcodes(seqs, upstream="AAAAA")) == 0

print("all checks passed")

#### One more number worth reporting: the __match rate__

Tests tell you that the pattern behaves correctly on cases you thought of. They cannot tell you
whether it is right for your __real__ data, because real data contains things you did not think of.
The number that tells you that is the fraction of sequences that matched:

- 95% matching: plausible, keep going
- 3% matching: your pattern is probably wrong, not your experiment
- 100% matching: suspicious - is your pattern strict enough to reject anything at all?

On Thursday we will compute this on ~10,000 real sequencing reads.

In [ ]:
matched = sum(count_barcodes(seqs).values())

print(f"{matched} of {len(seqs)} sequences matched the expected design "
      f"({100 * matched / len(seqs):.1f}%)")

----
# __Wrapping up__

1. Biological data arrives as semi-structured text, and pulling the structure out of it is a routine
   part of the work.
2. A pattern search is just __scan, test, report__. Regular expressions are a compact way to write
   down the __test__ so that the `re` module can do the scanning.
3. Describing the shape you want - in plain English, before any code - is the hard part. The syntax
   is the easy part, and it is well documented.
4. A wrong pattern fails silently, so write test cases (including __negative__ ones), keep them next
   to your code, and report your match rate.

#### On Thursday (lecture 09)

We will use the `biopython` module to read a real FASTQ file from an influenza sequencing run, and
apply exactly this workflow to several thousand real reads. __Homework 4 extends that analysis__, so
Thursday's notebook is the one to keep close.